# Part 1 (2022)

In [19]:
import http.client
import requests

conn = http.client.HTTPSConnection("v3.football.api-sports.io")

headers = {
    'x-apisports-key': "6f5377e8f97b90c8a25d661a945e599c"
    }

conn.request("GET", "/fixtures?league=1&season=2022", headers=headers)

res = conn.getresponse()
data = res.read()

#print(data.decode("utf-8"))

In [12]:
import pandas as pd
from datetime import datetime
import json

def clean_fixture_data(raw_json):
    rows = []

    for match in raw_json.get("response", []):
        fixture = match.get("fixture", {})
        league = match.get("league", {})
        teams = match.get("teams", {})
        goals = match.get("goals", {})
        score = match.get("score", {})

        home = teams.get("home", {})
        away = teams.get("away", {})

        halftime = score.get("halftime", {})
        fulltime = score.get("fulltime", {})
        extratime = score.get("extratime", {})
        penalty = score.get("penalty", {})

        # Build row
        row = {
            # Fixture
            "fixture_id": fixture.get("id"),
            "date": pd.to_datetime(fixture.get("date")),
            "timestamp": fixture.get("timestamp"),
            "venue": fixture.get("venue", {}).get("name"),
            "city": fixture.get("venue", {}).get("city"),
            "status": fixture.get("status", {}).get("short"),

            # League
            "league_id": league.get("id"),
            "season": league.get("season"),
            "round": league.get("round"),

            # Teams
            "home_team": home.get("name"),
            "away_team": away.get("name"),
            "home_id": home.get("id"),
            "away_id": away.get("id"),
            "home_win": int(home.get("winner")) if home.get("winner") is not None else None,
            "away_win": int(away.get("winner")) if away.get("winner") is not None else None,

            # Goals
            "home_goals": goals.get("home"),
            "away_goals": goals.get("away"),

            # Score breakdown
            "ht_home": halftime.get("home"),
            "ht_away": halftime.get("away"),
            "ft_home": fulltime.get("home"),
            "ft_away": fulltime.get("away"),
            "et_home": extratime.get("home"),
            "et_away": extratime.get("away"),
            "pen_home": penalty.get("home"),
            "pen_away": penalty.get("away"),
        }

        # Feature engineering
        if row["home_goals"] is not None and row["away_goals"] is not None:
            row["goal_diff"] = row["home_goals"] - row["away_goals"]
            row["total_goals"] = row["home_goals"] + row["away_goals"]

            if row["goal_diff"] > 0:
                row["result"] = 1
            elif row["goal_diff"] < 0:
                row["result"] = -1
            else:
                row["result"] = 0
        else:
            row["goal_diff"] = None
            row["total_goals"] = None
            row["result"] = None

        row["is_draw"] = 1 if row["result"] == 0 else 0
        row["went_to_et"] = 1 if row["et_home"] is not None else 0
        row["went_to_pen"] = 1 if row["pen_home"] is not None else 0

        rows.append(row)

    df = pd.DataFrame(rows)

    # Final cleanup
    df = df.sort_values("date")
    df.reset_index(drop=True, inplace=True)

    return df

In [14]:
clean_fixture_data(json.loads(data.decode("utf-8"))).to_csv(f"fixtures_output_2022.csv", index=False)

# Part 2 (1930-2014)

In [17]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re
import time

# =========================
# FUNCTION TO PARSE PAGE
# =========================
def parse_world_cup_html(html_content, source_url, year):
    soup = BeautifulSoup(html_content, "html.parser")
    
    pre = soup.find("pre")
    if not pre:
        return []
    
    text = pre.get_text("\n")
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    
    rows = []
    current_group = None
    
    for line in lines:
        # Detect group headers
        group_match = re.search(r'Group [A-D]', line)
        if group_match:
            current_group = group_match.group()
        
        # Detect match lines
        match_match = re.search(r'([A-Z]{3})\s*-\s*([A-Z]{3})\s*(\d+:\d+)', line)
        if match_match:
            rows.append({
                "year": year,
                "group": current_group,
                "team1": match_match.group(1),
                "team2": match_match.group(2),
                "score": match_match.group(3),
                "source_url": source_url
            })
    
    return rows


# =========================
# YOUR DATA
# =========================
urls = [
    "https://www.rsssf.org/tables/30full.html",
    "https://www.rsssf.org/tables/34full.html",
    "https://www.rsssf.org/tables/38full.html",
    "https://www.rsssf.org/tables/50full.html",
    "https://www.rsssf.org/tables/54full.html",
    "https://www.rsssf.org/tables/58full.html",
    "https://www.rsssf.org/tables/62full.html",
    "https://www.rsssf.org/tables/66full.html",
    "https://www.rsssf.org/tables/70full.html",
    "https://www.rsssf.org/tables/74full.html",
    "https://www.rsssf.org/tables/78full.html",
    "https://www.rsssf.org/tables/82full.html",
    "https://www.rsssf.org/tables/86full.html",
    "https://www.rsssf.org/tables/90full.html",
    "https://www.rsssf.org/tables/94full.html",
    "https://www.rsssf.org/tables/98full.html",
    "https://www.rsssf.org/tables/2002full.html",
    "https://www.rsssf.org/tables/2006full.html",
    "https://www.rsssf.org/tables/2010full.html",
    "https://www.rsssf.org/tables/2014full.html"
]

years = [
    1930,
    1934,
    1938,
    1950,
    1954,
    1958,
    1962,
    1966,
    1970,
    1974,
    1978,
    1982,
    1986,
    1990,
    1994,
    1998,
    2002,
    2006,
    2010,
    2014
]


# =========================
# MAIN LOOP
# =========================
all_rows = []

for url, year in zip(urls, years):
    try:
        response = requests.get(url)
        
        if response.status_code == 200:
            html_content = response.text
            parsed_rows = parse_world_cup_html(html_content, url, year)
            all_rows.extend(parsed_rows)
        else:
            print(f"Failed to retrieve {url} (Status: {response.status_code})")
        
        time.sleep(1)  # be polite to server
    
    except Exception as e:
        print(f"Error processing {url}: {e}")


# =========================
# EXPORT TO CSV
# =========================
df = pd.DataFrame(all_rows)

df.to_csv("world_cup_all_matches.csv", index=False)

print("CSV created: world_cup_all_matches.csv")

CSV created: world_cup_all_matches.csv
